**Capgemini Data Engineering Interview Questions**

**Position:** Data Engineer
**Experience:** 3–5 years
**Application Process:** Referral

### 🟡 Round 1 - Technical 1 (SQL | Python | Data Engineering Basics)

1. Write a SQL query to find duplicate records and remove them efficiently.

2. Difference between `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` with real examples.

3. Given a large dataset, how would you optimize a slow-running SQL query?

4. Python question: How do you handle missing or null values in a dataset?

5. Write a Python/PySpark script to read a large CSV file and process it efficiently.

6. What are different types of joins in SQL and when would you use each?



In [0]:
# 1. Write a SQL query to find duplicate records and remove them efficiently.

WITH duplicates AS (
    SELECT
        id,
        ROW_NUMBER() OVER (
            PARTITION BY name, email
            ORDER BY id
        ) AS rn
    FROM employees
)
DELETE FROM duplicates
WHERE rn > 1;

## 2. `ROW_NUMBER()` vs `RANK()` vs `DENSE_RANK()`

> **“All three are SQL window functions used to assign rankings to rows, but the main difference is how they handle duplicate or tied values.**
>
> **`ROW_NUMBER()`** assigns a unique sequential number to every row. Even if two employees have the same salary, they will get different row numbers.
>
> **`RANK()`** gives the same rank to rows with the same value, but it skips the next rank after a tie. For example, if two employees are ranked 2nd, the next employee will be ranked 4th.
>
> **`DENSE_RANK()`** also gives the same rank to tied values, but it doesn't skip any rank. So after two employees get rank 2, the next employee gets rank 3.
>
> For example, if salaries are **100K, 90K, 90K, and 80K**, the results would be:
>
> `ROW_NUMBER()` → **1, 2, 3, 4**
> `RANK()` → **1, 2, 2, 4**
> `DENSE_RANK()` → **1, 2, 2, 3**
>
> In real projects, I use `ROW_NUMBER()` mainly for **deduplication and selecting a single record**, `RANK()` when **ranking with gaps** is required, and `DENSE_RANK()` when I need **ranking with ties but without gaps**, such as finding the second or third highest salary.”**

### ⭐ One-line version for a quick interview

**ROW_NUMBER = unique ranking | RANK = ranking with gaps | DENSE_RANK = ranking without gaps.**


## 3. How would you optimize a slow-running SQL query on a large dataset?

> **“First, I would identify the root cause rather than immediately changing the query. I would check the execution plan to understand whether the query is doing full table scans, expensive joins, sorting, or aggregations.**
>
> **Then I would optimize it by creating appropriate indexes on columns used frequently in `WHERE`, `JOIN`, and `ORDER BY` clauses. I would avoid `SELECT *` and retrieve only the required columns.**
>
> **I would also filter the data as early as possible, optimize joins, avoid unnecessary subqueries and correlated queries, and make sure statistics are up to date. For very large tables, I would consider partitioning the data based on commonly filtered columns such as date.**
>
> **I would also check for data skew and inefficient joins, and if necessary, rewrite the query or use techniques such as pre-aggregation or materialized views.**
>
> **Finally, I would compare the execution plan and query execution time before and after the changes to confirm that the optimization actually improved performance.”**

### 🔥 Key points to remember

**Execution Plan → Indexing → Filtering → Joins → Avoid `SELECT *` → Partitioning → Statistics → Data Skew → Validate Performance**

### Example

Instead of:

```sql
SELECT *
FROM orders
WHERE YEAR(order_date) = 2026;
```

Prefer:

```sql
SELECT order_id, customer_id, order_date, amount
FROM orders
WHERE order_date >= '2026-01-01'
  AND order_date < '2027-01-01';
```

The second approach avoids applying a function to the column and allows the database to make better use of an index on `order_date`.

**Best closing line in an interview:**

> **“First analyze the execution plan, identify the bottleneck, make targeted changes, and then measure the improvement.”**


In [0]:
# 4. Handle missing or null values

from pyspark.sql.functions import col

# Check null count
df.select(
    "employee_id",
    "salary",
    "department"
).show()

# Fill missing values
df = df.fillna({
    "salary": 0,
    "department": "Unknown"
})

# Remove records where employee_id is null
df = df.filter(col("employee_id").isNotNull())

## 5. Read and Process a Large CSV Using PySpark

> **“For a large CSV file, I would use PySpark instead of Pandas because Spark can distribute the processing across multiple executors. I would read only the required columns, define the schema explicitly instead of relying on schema inference, filter unnecessary records as early as possible, and avoid collecting the data to the driver. I would also use an appropriate number of partitions and write the processed data in a distributed format such as Delta or Parquet.”**

### PySpark Example

```python
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("LargeCSVProcessing") \
    .getOrCreate()

# Define schema explicitly
schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True)
])

# Read CSV
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("/data/input/employees.csv")

# Select only required columns
df = df.select(
    "employee_id",
    "name",
    "department",
    "salary"
)

# Filter early
df = df.filter(
    col("salary").isNotNull() &
    (col("salary") > 50000)
)

# Repartition if required for downstream processing
df = df.repartition("department")

# Write in an efficient columnar format
df.write \
    .mode("overwrite") \
    .partitionBy("department") \
    .parquet("/data/output/employees")
```

### 🚀 How I would optimize it

1. **Explicit schema** → avoids expensive schema inference.
2. **Select required columns** → reduces data read and memory usage.
3. **Filter early** → reduces the amount of data processed.
4. **Avoid `collect()` / `toPandas()`** → prevents driver memory issues.
5. **Use Parquet/Delta instead of CSV** for downstream processing.
6. **Partition appropriately** → improves parallel processing and query performance.
7. **Avoid unnecessary repartitioning** → repartition causes a shuffle.
8. **Use `coalesce()`** when reducing partitions without requiring a full shuffle.

### ⭐ Strong interview closing

> **“For very large CSV files, my main focus would be minimizing data movement and avoiding driver-side processing. I would use explicit schemas, column pruning, predicate pushdown where applicable, appropriate partitioning, and write the processed data to Delta or Parquet for efficient future reads.”**


6. What are different types of joins in SQL and when would you use each?


| Join                | What it returns            | Common use                            |
| ------------------- | -------------------------- | ------------------------------------- |
| **INNER JOIN**      | Matching rows only         | Get matching data                     |
| **LEFT JOIN**       | All left + matching right  | Keep all records from primary table   |
| **RIGHT JOIN**      | All right + matching left  | Keep all records from secondary table |
| **FULL OUTER JOIN** | All rows from both         | Data reconciliation                   |
| **CROSS JOIN**      | Every possible combination | Generate combinations                 |
| **SELF JOIN**       | Table joined to itself     | Hierarchical relationships            |


### 🔵 Round 2 - Technical 2 (ADF | Azure | Databricks)

1. How do you design an end-to-end ETL pipeline in Azure Data Factory?

2. Explain incremental data loading strategies in ADF.

3. What is partitioning in Databricks and how does it improve performance?

4. Difference between Data Lake vs Data Warehouse vs Lakehouse architecture.

5. How does Delta Lake ensure ACID transactions?

6. How do you monitor and troubleshoot failures in ADF pipelines?




## 1. Azure Data Factory — End-to-End ETL Pipeline Steps

1. **Identify Source & Target** → Define data sources and destination.
2. **Create Linked Services** → Connect ADF to source and target systems.
3. **Create Datasets** → Define source and target data structures.
4. **Build Pipeline** → Use Copy Activity for data ingestion.
5. **Transform Data** → Use Mapping Data Flow or Databricks.
6. **Implement Incremental Load** → Use watermark/CDC logic.
7. **Add Parameters & Variables** → Make pipelines reusable and dynamic.
8. **Add Error Handling** → Configure retries, failure paths, and logging.
9. **Add Triggers** → Schedule or event-based execution.
10. **Monitor & Optimize** → Monitor runs, troubleshoot failures, and optimize performance.


## 2. Incremental Loading in ADF — Brief Explanation

**1. Watermark / Last Modified Date**
Store the last successful load timestamp and fetch only records modified after that time.

**2. CDC (Change Data Capture)**
Captures **INSERT, UPDATE, and DELETE** changes and loads only those changes.

**3. Change Tracking**
Uses database change-tracking information to identify modified records since the last load.

**4. High-Watermark ID**
Store the last processed ID and load records where `ID > LastProcessedID`.

**5. File-Based Incremental Load**
Process only newly arrived or modified files instead of reprocessing all files.

**6. Metadata-Driven Load**
Maintain a control table containing source table, watermark, load status, and other configuration details to make pipelines reusable.

**7. Merge / Upsert**
Insert new records and update existing records in the target based on a business key.

### ⭐ Interview Answer

> **“For incremental loading in ADF, I usually use a watermark-based approach. I maintain the last successful timestamp or ID in a control table, extract only new or modified records, load them into the target, and update the watermark after successful completion.”**


## 3. Partitioning in Databricks — Brief Interview Answer

> **“Partitioning is a technique of dividing a large dataset into smaller subsets based on one or more columns, such as date, region, or department. In Databricks, partitioned data is physically organized into separate directories, which allows Spark to read only the relevant partitions instead of scanning the entire dataset.”**

### Example

```python
df.write \
  .partitionBy("year", "month") \
  .format("delta") \
  .save("/data/sales")
```

If we query:

```sql
SELECT *
FROM sales
WHERE year = 2026 AND month = 8;
```

Spark can perform **partition pruning** and read only the relevant partition.

### How it improves performance

* **Partition pruning** → Reads only required partitions.
* **Less I/O** → Avoids scanning unnecessary data.
* **Faster queries** → Especially for large datasets.
* **Better parallelism** → Data can be processed across Spark partitions.

### ⭐ Interview Tip

> **“I choose partition columns that are frequently used in filtering, such as date, and avoid high-cardinality columns like customer ID because they can create too many small files and partitions.”**


## 4. Data Lake vs Data Warehouse vs Lakehouse — Interview Answer

> **“A Data Lake stores large volumes of raw structured, semi-structured, and unstructured data at low cost. A Data Warehouse stores cleaned and structured data mainly for reporting and BI. A Lakehouse combines the flexibility and scalability of a Data Lake with the reliability, governance, and SQL analytics capabilities of a Data Warehouse.”**

| Feature    | Data Lake         | Data Warehouse     | Lakehouse               |
| ---------- | ----------------- | ------------------ | ----------------------- |
| Data       | Raw + all formats | Structured         | Raw + structured        |
| Processing | ETL/ELT           | ETL/ELT            | ETL/ELT                 |
| Cost       | Low               | Higher             | Relatively low          |
| Analytics  | Limited/varied    | Excellent BI       | BI + ML + Analytics     |
| ACID       | Usually limited   | Yes                | Yes                     |
| Example    | ADLS, S3          | Snowflake, Synapse | Databricks + Delta Lake |

### ⭐ Easy way to remember

**Data Lake → Store everything**
**Data Warehouse → Analyze structured data**
**Lakehouse → Store + Process + Analyze everything in one platform**


## 5. How does Delta Lake ensure ACID transactions?

> **“Delta Lake provides ACID transactions using its transaction log, also called the `_delta_log`. Every write operation is recorded as a transaction in this log, which allows Delta Lake to maintain consistency and provide reliable concurrent reads and writes.”**

**A — Atomicity:**
A transaction either **completes fully or doesn't commit**. Partial updates are not visible.

**C — Consistency:**
Delta Lake maintains data integrity through **schema enforcement and transaction validation**.

**I — Isolation:**
Concurrent operations don't interfere with each other. Readers see a **consistent snapshot** of the table using **snapshot isolation**.

**D — Durability:**
Once a transaction is committed, the changes are persisted in the underlying cloud storage such as **ADLS or S3**.

### ⭐ Interview Shortcut

**Atomicity → All or nothing**
**Consistency → Data remains valid**
**Isolation → Concurrent transactions don't conflict**
**Durability → Committed data persists**

**Best interview line:**

> **“Delta Lake achieves ACID transactions primarily through its transaction log, which tracks every table change and enables atomic commits, consistent snapshots, concurrency control, and reliable recovery.”**


## 6. How do you monitor and troubleshoot failures in ADF pipelines?

### Short Steps

1. **Monitor** → Check Pipeline Runs in ADF Monitor.
2. **Identify** → Find the failed activity.
3. **Analyze** → Check error message, input/output, and logs.
4. **Validate** → Check connectivity, permissions, parameters, and data.
5. **Retry** → Configure retries for transient failures.
6. **Fix** → Resolve the root cause.
7. **Rerun** → Restart the failed pipeline/activity.
8. **Alert** → Configure Azure Monitor alerts/notifications.

### ⭐ Interview Shortcut

**Monitor → Identify → Analyze → Validate → Fix → Retry/Rerun → Alert**


### 🔴 Round 3 - Managerial / Behavioral

1. Tell me about a challenging data engineering project you worked on.

2. How do you handle tight deadlines and multiple...